### Unificación de datos

El siguiente notebook se encarga de mezclar todos los archivos csv de datos de buses y metro, para crear dos nuevos archivos csv con estos datos unificados.

In [33]:
import pandas as pd
import pandas.errors
from os import listdir
from os.path import join

buses_csvs = listdir(join("..", "buses_outputs"))
metro_csvs = listdir(join("..", "metro_outputs"))

if ".ipynb_checkpoints" in buses_csvs:
    buses_csvs.remove(".ipynb_checkpoints")
if ".ipynb_checkpoints" in metro_csvs:
    metro_csvs.remove(".ipynb_checkpoints")

unified_bus_df_list = []
unified_metro_df_list = []

for bus_csv in buses_csvs:
    bus_csv_df = pd.read_csv(join("..", "buses_outputs", bus_csv))
    unified_bus_df_list.append(bus_csv_df)

for metro_csv in metro_csvs: # Habemus un problemón los datos desde el 25-10-25 están bugueados
    try:
        metro_csv_df = pd.read_csv(join("..", "metro_outputs", metro_csv))
    except pandas.errors.EmptyDataError:
        continue
    unified_metro_df_list.append(metro_csv_df)

unified_bus_df = pd.concat(unified_bus_df_list, ignore_index=True, sort=False)
unified_metro_df = pd.concat(unified_metro_df_list, ignore_index=True, sort=False)
unified_bus_df.drop(columns=["X", "Y"], inplace=True)

Añadiremos datos de cada paradero al dataframe.

In [34]:
import json
# Abrimos el archivo donde tenemos información importantes de los pasajeros
ruta_json = join("..", "data", "Paraderos-Santiago-Chile.geojsonl.json")
# Guardamos su informacion en una lista
datos = []
with open(ruta_json, "r", encoding="utf-8-sig") as f:
    for archivo in f:
        datos.append(json.loads(archivo.strip()))
# creamos un datafraem con los datos de la lista
df_paradero = pd.json_normalize(datos)
# Renombramos columnas que estarán en nuestro DataFrame
df_paradero = df_paradero.rename(columns={"properties.ID": "id", 
                            "properties.CODINFRA": "codinfra",
                            "properties.SIMT": "bus_stop_code",
                            "properties.COMUNA": "comuna",
                            "properties.NOMBRE_PAR": "nombre_par",
                            "properties.NSERVICIOS": "n_servicios",
                            "properties.SERVICIOS": "servicios",
                            "geometry.coordinates": "coordinates"})
# Seleccionamos dichas columnas
df_paradero = df_paradero[["id", "codinfra", "comuna", "bus_stop_code", "nombre_par", "n_servicios", "servicios", "coordinates"]]
# Reemplazamos nombres de comunas pues el archivo venía con un encoding incorrecto
df_paradero["comuna"] = df_paradero["comuna"].replace(
    {"CONCHAL�": "CONCHALÍ",
    "ESTACI�N CENTRAL": "ESTACIÓN CENTRAL",
    "MAIP�": "MAIPÚ",
    "PE�ALOL�N": "PEÑALOLÉN",
    "SAN JOAQU�N": "SAN JOAQUÍN",
    "SAN RAM�N": "SAN RAMÓN",
    "�U�OA": "ÑUÑOA"
    })
df_paradero.head()
df_paradero_copy = df_paradero.copy()
df_paradero_copy["servicios"] = df_paradero_copy["servicios"].str.replace(r"([IR])(?=;|$)", "", regex=True)
df_paradero_copy.to_csv(join("..", "data", "paraderos.csv"), index=False)

Reparamos las coordenadas de cada paradero.

In [35]:
unified_bus_df = pd.merge(unified_bus_df, df_paradero, on= "bus_stop_code", how="inner")
unified_bus_df[["lan", "lon"]] = pd.DataFrame(unified_bus_df["coordinates"].tolist(), index=unified_bus_df.index)
unified_bus_df.drop(columns=["coordinates"], inplace= True)

Podemos notar al revisar este DataFrame, en la columna 'servicios', que todos los códigos de micro tienen una letra al final (Sabemos que puede ser N, C, E, V) pero también las hay en casos que no corresponden. En consecuencia, es necesario que los datos correspondientes a los recorridos sean los correctos-

In [36]:
clean_unified_bus_df = unified_bus_df.copy()
clean_unified_bus_df["servicios"] = clean_unified_bus_df["servicios"].str.replace(r"([IR])(?=;|$)", "", regex=True)
print(clean_unified_bus_df.head())
clean_unified_bus_df["date"] = pd.to_datetime(clean_unified_bus_df["date"])
clean_unified_bus_df["hour_minute"] = clean_unified_bus_df["date"].dt.strftime('%H:%M')
clean_unified_bus_df["date_only"] = clean_unified_bus_df["date"].dt.strftime("%Y-%m-%d")
clean_unified_bus_df = clean_unified_bus_df.drop(columns=["date"])
#print(clean_unified_bus_df.head())

  bus_stop_code route_id   bus_id  meters_distance  min_arrival_time  \
0          PC96      216  VJWZ-95             4427                12   
1          PC96      216  SFWZ-21            11100                35   
2         PB169      B15  SPCG-69             5751                16   
3         PB169      116  LDJJ-64             4724                13   
4         PA838      B20  LXDH-88             4328                12   

   max_arrival_time                        date    id       codinfra  \
0                16  2025-11-01 06:00:03.990521  1244  L-17-16-25-SN   
1                60  2025-11-01 06:00:03.990528  1244  L-17-16-25-SN   
2                20  2025-11-01 06:00:08.033221  9567   L-4-23-15-SN   
3                17  2025-11-01 06:00:08.033228  9567   L-4-23-15-SN   
4                16  2025-11-01 06:00:10.071736  9186  T-20-131-PO-4   

       comuna                                nombre_par  n_servicios  \
0  LAS CONDES  Av. Manquehue Norte / esq. Los Militares       

Por último, exportamos los dataframes en csv a la carpeta csv_unificado_{servicio}.

In [37]:
clean_unified_bus_df.to_csv(join("..", "csv_unificado_buses", "unified_bus_data.csv"), index=False)
unified_metro_df.to_csv(join("..", "csv_unificado_metro", "unified_metro_data.csv"), index=False)

In [38]:
# Creamos DataFrame que calcula la densidad poblacional:
df_densidad = pd.read_csv('../data/censo_proyecciones_año.csv', sep = ";")
df_densidad = df_densidad[df_densidad['region'] == "Metropolitana de Santiago"]
df_densidad = df_densidad[df_densidad['año'] == 2025]
df_densidad.columns

Index(['Unnamed: 0', 'cut_region', 'region', 'cut_provincia', 'provincia',
       'cut_comuna', 'comuna', 'año', 'población'],
      dtype='object')

In [39]:
# datos sacados de https://archplg.cl/geografia-y-ocupacion-de-la-cuenca-de-santiago/ y
# https://lobarnechea.cl/pladeco/etapa-1/
comunas_km2 = {
    'Padre Hurtado': 81,
    'Pudahuel': 197,
    'San Bernardo': 153,
    'Huechuraba': 45,
    'Las Condes': 99,
    'Vitacura': 28,
    'Quilicura': 57,
    'Maipú': 138,
    'La Reina': 23,
    'Peñalolén': 54,
    'Cerrillos': 17,
    'La Florida': 71,
    'Renca': 24,
    'Puente Alto': 88,
    'Macul': 13,
    'La Cisterna': 10,
    'Quinta Normal': 12,
    'San Joaquín': 10,
    'Recoleta': 16,
    'Providencia': 14,
    'Estación Central': 14,
    'San Miguel': 10,
    'Pedro Aguirre Cerda': 9,
    'Conchalí': 11,
    'El Bosque': 14,
    'La Granja': 10,
    'Cerro Navia': 11,
    'Ñuñoa': 17,
    'Lo Espejo': 8,
    'Lo Prado': 7,
    'San Ramón': 6,
    'Independencia': 7,
    'Lo Barnechea': 49
}
comunas_lista = list(comunas_km2.keys())
densidad_2025 = df_densidad[df_densidad['comuna'].isin(comunas_lista)].copy()
densidad_2025['km^2'] = densidad_2025['comuna'].map(comunas_km2)

In [40]:
len(densidad_2025['comuna'])

33

In [41]:
densidad_2025['densidad_poblacional'] = round(densidad_2025['población'] / densidad_2025['km^2'])
densidad_2025.drop(columns=['provincia', 'region'], inplace=True)
densidad_2025.head(2)

,Unnamed: 0,cut_region,cut_provincia,cut_comuna,comuna,año,población,km^2,densidad_poblacional
8795,8796,13,131,13102,Cerrillos,2025,90268,17,5310.0
8829,8830,13,131,13103,Cerro Navia,2025,138677,11,12607.0


In [42]:
densidad_2025.to_csv(join("..", "data", "densidad_2025.csv"), index=False)